# 01 — Data Preprocessing
**Dataset:** Fruits 360  
**Task:** Image Classification menggunakan Machine Learning  
**Tujuan Notebook:** Load dataset, eksplorasi awal, resize, normalisasi, dan simpan fitur.

## 1. Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Konfigurasi Path Dataset

In [ ]:
# Path dataset di drive D
TRAIN_DIR  = r'D:\fruits-360_100x100\fruits-360\Training'
TEST_DIR   = r'D:\fruits-360_100x100\fruits-360\Test'
OUTPUT_DIR = 'data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = (100, 100)  # Gambar Fruits-360 sudah 100x100

# Ambil daftar kelas (folder = label)
classes = sorted(os.listdir(TRAIN_DIR))
print(f'Total kelas  : {len(classes)}')
print(f'Contoh kelas : {classes[:10]}')

## 3. Eksplorasi Dataset

In [ ]:
# Hitung jumlah gambar per kelas
class_counts = {}
for cls in classes:
    cls_path = os.path.join(TRAIN_DIR, cls)
    class_counts[cls] = len(os.listdir(cls_path))

total_images = sum(class_counts.values())
print(f'Total gambar training  : {total_images}')
print(f'Jumlah kelas           : {len(classes)}')
print(f'Rata-rata per kelas    : {total_images // len(classes)}')

# Visualisasi distribusi 20 kelas pertama
top_classes = dict(list(class_counts.items())[:20])
plt.figure(figsize=(14, 5))
plt.bar(top_classes.keys(), top_classes.values(), color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Jumlah Gambar per Kelas (20 Kelas Pertama)')
plt.ylabel('Jumlah Gambar')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=100)
plt.show()

In [ ]:
# Tampilkan contoh gambar dari 9 kelas pertama
sample_classes = classes[:9]
fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for ax, cls in zip(axes.flatten(), sample_classes):
    cls_path = os.path.join(TRAIN_DIR, cls)
    img_file = os.listdir(cls_path)[0]
    img = Image.open(os.path.join(cls_path, img_file))
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis('off')

plt.suptitle('Contoh Gambar dari Berbagai Kelas', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=100)
plt.show()

## 4. Preprocessing: Resize & Normalisasi

In [ ]:
def load_images_from_folder(folder, classes, img_size=(100, 100), max_per_class=None):
    """
    Load gambar dari folder, resize, dan normalisasi pixel ke [0,1].
    Returns: X (array gambar), y (label integer), label_names (list nama kelas)
    """
    X, y = [], []
    label_map = {cls: idx for idx, cls in enumerate(classes)}

    for cls in classes:
        cls_path = os.path.join(folder, cls)
        if not os.path.isdir(cls_path):
            continue
        files = os.listdir(cls_path)
        if max_per_class:
            files = files[:max_per_class]

        for fname in files:
            fpath = os.path.join(cls_path, fname)
            try:
                img = Image.open(fpath).convert('RGB')
                img = img.resize(img_size)
                img_array = np.array(img) / 255.0  # Normalisasi ke [0, 1]
                X.append(img_array)
                y.append(label_map[cls])
            except Exception as e:
                print(f'Error loading {fpath}: {e}')

    return np.array(X), np.array(y), classes

# max_per_class=50 agar proses cepat.
# Hapus parameter max_per_class jika ingin load semua gambar (lebih lama).
print('Loading training data...')
X_train, y_train, label_names = load_images_from_folder(
    TRAIN_DIR, classes, IMG_SIZE, max_per_class=50
)

print('Loading test data...')
X_test, y_test, _ = load_images_from_folder(
    TEST_DIR, classes, IMG_SIZE, max_per_class=20
)

print(f'\nX_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'Jumlah kelas  : {len(set(y_train))}')

## 5. Flatten untuk ML Klasik

In [ ]:
# Flatten gambar dari (N, H, W, C) ke (N, H*W*C) untuk input ke model ML
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

print(f'X_train_flat shape : {X_train_flat.shape}')
print(f'X_test_flat  shape : {X_test_flat.shape}')
print(f'Ukuran fitur       : {X_train_flat.shape[1]} (= 100x100x3)')

## 6. Simpan Data

In [ ]:
# Simpan array ke file .npy untuk digunakan di notebook selanjutnya
np.save(f'{OUTPUT_DIR}/X_train_flat.npy', X_train_flat)
np.save(f'{OUTPUT_DIR}/X_test_flat.npy',  X_test_flat)
np.save(f'{OUTPUT_DIR}/y_train.npy',      y_train)
np.save(f'{OUTPUT_DIR}/y_test.npy',       y_test)

# Simpan daftar label
pd.Series(label_names).to_csv(
    f'{OUTPUT_DIR}/label_names.csv', index=True, header=['class_name']
)

print('Data berhasil disimpan ke folder data/processed/')
print(f'   - X_train_flat : {X_train_flat.shape}')
print(f'   - X_test_flat  : {X_test_flat.shape}')
print(f'   - y_train      : {y_train.shape}')
print(f'   - y_test       : {y_test.shape}')
print(f'   - label_names  : {len(label_names)} kelas')

## 7. Ringkasan

| Langkah | Detail |
|---|---|
| Dataset | Fruits 360 (100x100 px) |
| Jumlah kelas | 131 kelas buah |
| Gambar training | 50 per kelas (max_per_class=50) |
| Gambar test | 20 per kelas (max_per_class=20) |
| Normalisasi | Pixel dibagi 255, rentang [0, 1] |
| Output | Flatten vektor 30.000 dimensi (100x100x3) |

Lanjut ke notebook 02_feature_engineering.ipynb